<a href="https://colab.research.google.com/github/yiyu-chen-labs/llm-from-scratch/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG Baseline 探索筆記 — JQaRA (純文字檢索)

## 目標
先用純文字資料(JQaRA)熟悉 RAG 檢索的整條 workflow
專注在「檢索」這一半:向量化 → 相似度排序 → 用 label 驗證準不準

## 資料:JQaRA (hotchpotch/JQaRA)
- 日文 RAG 評估資料集,知識庫是 Wikipedia 段落(純文字,無圖無 PDF)
- 結構:資料是"flat"的,一題的每個候選段落各佔一列,用"q_id"binding
- 關鍵欄位:question(問題)、text(候選段落)、answers(正解)、
  label(1=正解段落 / 0=干擾項)、q_id(綁定同題的鑰匙)
- label 是"評估用的答案卡",系統實際檢索時看不到它

## 做了什麼
1. 用 q_id 篩出一題 + 它的所有候選段落
2. embedding 模型:intfloat/multilingual-e5-small(384 維,支援日文)
3. 把 question 和候選段落向量化,算 cosine similarity,由高到低排序
4. 用 label 驗證:相似度排名前面的,是不是 label=1 的正解

## 關鍵觀察
- 檢索**大致有效**:正解(label=1)被撈進前 3 名(排第 2、第 3)
- 但**排序不完美**:第 1 名是一個 label=0 的干擾段落(海蛞蝓/ウミウシ,
  相似度 0.690),它擠掉了正解
- **原因**:問題問「海の天使」(クリオネ),干擾段落提到「海のナメクジ」,
  兩者**語意極相近**,embedding 因此誤判它最相關
- **結論**:embedding 找的是「語意最相似」,不等於「含正解」
  **相似 ≠ 正確** — 這是 RAG 的核心難題,我親眼看到它發生

## 這對 Phase 2 的意義
- **Re-ranking (Week 11)**:baseline 把相關段落撈進來了造成排序瑕疵
  (干擾項排第一)→ 需要更精細的模型重排 and this is why we need re-ranking
- **引用忠實度 / abstention (Week 12)**:若 LLM 拿到第一名那個無關段落作答,可能答錯 → 需要檢查「答案是否真被段落支持」的機制

## 待辦 / 下一步
- [ ] 玩法 B:自己建 FAISS 向量庫,從整庫檢索(非現成候選)。
- [ ] 把「正解平均排第幾」算成量化指標(MRR / recall@k)當評估基準。
- [ ] chunking 練習需另找「原始長文件」資料集(JQaRA 段落已切好,練不到 chunking)。
- [ ] 生成那半(LLM 讀 prompt 作答)待接 API 後做。

In [ ]:
from datasets import load_dataset

ds = load_dataset("hotchpotch/JQaRA")
print(ds)          # 看有哪些 split、多少筆、欄位有哪些

In [ ]:
sample = ds['dev'][0]
print(sample)

In [ ]:
sample = ds['dev'][0]
for key, value in sample.items():
    print(f"{key}: {value}")
    print("-" * 40)

In [ ]:
from pprint import pprint
pprint(sample)

In [ ]:
import pandas as pd

df = ds['dev'].to_pandas()

print("總筆數", len(df))
print("不同問題數", df['q_id'].nunique())

first_qid = df['q_id'].iloc[0]
one_question = df[df['q_id']==first_qid]


print("\n這一題的 q_id:", first_qid)
print("這一題有幾個候選段落:", len(one_question))
print("其中 label=1（正解）有幾個:", (one_question['label'] == 1).sum())


In [ ]:
first_qid = df['q_id'].iloc[0]

print("篩選前，df 總列數:", len(df))          # 幾萬
print("first_qid 是:", first_qid)

one_question = df[df['q_id'] == first_qid]
print("篩選後，one_question 列數:", len(one_question))  # 約100（只有這題）
print("篩選後，df 還是:", len(df), "列（沒變）")        # 還是幾萬，證明 df 沒被改

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

print("Model is Ready")
print("向量維度",model.get_sentence_embedding_dimension())

In [ ]:
question = one_question['question'].iloc[0]

passages = one_question['text'].tolist()
print("Question:",question)
print("Answer:",passages)

In [ ]:
q_vec = model.encode(question)
p_vecs = model.encode(passages)

print("\n問題向量的形狀:", q_vec.shape)   # (384,) → 一串 384 個數字
print("段落向量的形狀:", p_vecs.shape)    # (N, 384) → N 段，每段 384
print("\n問題向量前 5 個數字:", q_vec[:5]) # 偷看一下「文字變成的數字」長怎樣

In [ ]:
from sentence_transformers import util
import numpy as np


#cosine similarity
scores = util.cos_sim(q_vec, p_vecs)[0]
ranked_idx = np.argsort(scores.numpy())[::-1]


print("排名 | 相似度 | label | 段落前30字")
print("-" * 60)
for rank, i in enumerate(ranked_idx[:10]):
    label = one_question['label'].iloc[i]
    text_preview = passages[i][:30]
    mark = "✓正解" if label == 1 else ""
    print(f"{rank+1:>3}  | {scores[i]:.3f} | {label} {mark} | {text_preview}")

In [ ]:
# 找出所有正解(label==1)排在第幾名
for rank, i in enumerate(ranked_idx):
    if one_question['label'].iloc[i] == 1:
        print(f"正解排在第 {rank+1} 名 (相似度 {scores[i]:.3f})")